In [1]:
import rasterio
import numpy as np

# Path to your downloaded stack
tif_path = 'CycloneGabrielle_HawkesBay_S1_Stack.tif'

with rasterio.open(tif_path) as src:
    print(f"Raster Dimensions: {src.width} x {src.height} pixels")
    print(f"Number of Bands  : {src.count}")
    print(f"CRS / Projection : {src.crs}")
    print(f"Spatial Resolution: {src.res} meters")
    
    # Read all 6 bands
    # Band 1: pre_vv, Band 2: pre_vh, Band 3: post_vv, Band 4: post_vh, Band 5: vv_diff, Band 6: baseline_mask
    stack_data = src.read()
    
print("Data array shape (Bands, Height, Width):", stack_data.shape)

Raster Dimensions: 4330 x 6667 pixels
Number of Bands  : 6
CRS / Projection : EPSG:32760
Spatial Resolution: (10.0, 10.0) meters
Data array shape (Bands, Height, Width): (6, 6667, 4330)


In [2]:
import os
import rasterio
import numpy as np

# 1. Output Directories Setup
base_dir = 'dataset_patches'
images_dir = os.path.join(base_dir, 'images')
masks_dir = os.path.join(base_dir, 'masks')

os.makedirs(images_dir, exist_ok=True)
os.makedirs(masks_dir, exist_ok=True)

# 2. Patching Configuration
PATCH_SIZE = 256
STRIDE = 128  # 50% overlap to increase dataset size (data augmentation)

# 3. Read GeoTIFF
tif_path = 'CycloneGabrielle_HawkesBay_S1_Stack.tif'

with rasterio.open(tif_path) as src:
    # Bands 1-5: Input Features (pre_vv, pre_vh, post_vv, post_vh, vv_diff)
    features = src.read([1, 2, 3, 4, 5])  # Shape: (5, H, W)
    # Band 6: Target Mask (baseline_mask)
    mask = src.read(6)                    # Shape: (H, W)

_, height, width = features.shape

patch_count = 0

# 4. Sliding Window Tiling
for y in range(0, height - PATCH_SIZE + 1, STRIDE):
    for x in range(0, width - PATCH_SIZE + 1, STRIDE):
        
        # Extract patch
        feature_patch = features[:, y:y+PATCH_SIZE, x:x+PATCH_SIZE]
        mask_patch = mask[y:y+PATCH_SIZE, x:x+PATCH_SIZE]
        
        # Skip empty background patches (e.g., NaN or zero-data borders)
        if np.isnan(feature_patch).any() or np.all(feature_patch == 0):
            continue
            
        # Optional: Only keep patches that have meaningful land/flood content
        # Save feature patch (5 channels) and mask patch (1 channel) as NumPy arrays
        img_filename = os.path.join(images_dir, f'patch_{patch_count:04d}.npy')
        mask_filename = os.path.join(masks_dir, f'patch_{patch_count:04d}.npy')
        
        np.save(img_filename, feature_patch.astype(np.float32))
        np.save(mask_filename, mask_patch.astype(np.float32))
        
        patch_count += 1

print(f"Successfully generated {patch_count} patches of size {PATCH_SIZE}x{PATCH_SIZE}.")
print(f"Patches saved to folder: '{base_dir}/'")

Successfully generated 1632 patches of size 256x256.
Patches saved to folder: 'dataset_patches/'


In [4]:
import glob
from sklearn.model_selection import train_test_split

image_paths = sorted(glob.glob('dataset_patches/images/*.npy'))
mask_paths = sorted(glob.glob('dataset_patches/masks/*.npy'))

# Split into Train and Temp (Val + Test)
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    image_paths, mask_paths, test_size=0.30, random_state=42
)

# Split Temp into Validation and Test
val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs, temp_masks, test_size=0.50, random_state=42
)

print(f"Train Patches: {len(train_imgs)}")
print(f"Val Patches  : {len(val_imgs)}")
print(f"Test Patches : {len(test_imgs)}")

Train Patches: 1142
Val Patches  : 245
Test Patches : 245
